In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
print("Libraries loaded!")

In [ ]:
# 1. Load data
income_df = pd.read_csv(r'data\ACSST5Y2024.S1901-Data.csv', skiprows=[1])

# 2. DATA CLEANUP (Handle symbols BEFORE renaming/subsetting)
income_cols = ['S1901_C01_012E', 'S1901_C02_012E', 'S1901_C03_012E']

for col in income_cols:
    # Remove commas/plus signs and force to numeric
    income_df[col] = income_df[col].astype(str).str.replace(',', '').str.replace('+', '')
    income_df[col] = pd.to_numeric(income_df[col], errors='coerce')

# 3. Subset and Rename
income_df = income_df[['GEO_ID', 'S1901_C01_012E']].copy()
income_df.columns = ['geo_id', 'median_income']

# 4. Extract Zip Code and final drop of NaNs
income_df['zip_code'] = income_df['geo_id'].str.split('US').str[-1]
income_df = income_df.dropna(subset=['median_income'])

print(f"Loaded income data for {len(income_df)} Zip Codes.")

In [ ]:
# Load the tree data (You can start with 2nd parameter nrows=10000 to test, then remove nrows later)
trees_df = pd.read_csv('data/2015_Street_Tree_Census_-_Tree_Data_20260315.csv')

# 1. Filter to only 'Alive' trees (we can't measure the health of a stump)
trees_df = trees_df[trees_df['status'] == 'Alive'].copy()

# 2. Keep only the relevant columns
trees_df = trees_df[['tree_id', 'health', 'postcode', 'spc_common', 'borough']]

# 3. Ensure Zip Code (postcode) is a string to match the income data
trees_df['postcode'] = trees_df['postcode'].astype(str)

print(f"Loaded {len(trees_df)} living trees.")
trees_df.head()

In [ ]:
# --- 1. INCOME DATA CLEANUP ---
income_df = pd.read_csv(r'data\ACSST5Y2024.S1901-Data.csv', skiprows=[1])

income_cols = ['S1901_C01_012E', 'S1901_C02_012E', 'S1901_C03_012E']
for col in income_cols:
    income_df[col] = income_df[col].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False)
    income_df[col] = pd.to_numeric(income_df[col], errors='coerce')

income_df = income_df[['GEO_ID', 'S1901_C01_012E']].copy()
income_df.columns = ['geo_id', 'median_income']
income_df['zip_code'] = income_df['geo_id'].str.split('US').str[-1].astype(str)
income_df = income_df.dropna(subset=['median_income'])

# --- 2. TREE DATA PREPARATION ---
health_map = {'Good': 3, 'Fair': 2, 'Poor': 1}
trees_df['health_numeric'] = trees_df['health'].map(health_map)

# Filter for alive trees ONLY (handles your non-living tree question)
if 'status' in trees_df.columns:
    trees_alive = trees_df[trees_df['status'] == 'Alive'].copy()
else:
    trees_alive = trees_df.copy()

trees_alive['postcode'] = trees_alive['postcode'].astype(str)

# --- 3. TREE DATA AGGREGATION ---
# We use .agg() and then IMMEDIATELY rename the columns to match your later code
tree_stats = trees_alive.groupby('postcode').agg({
    'tree_id': 'count',
    'health_numeric': 'mean',
    'borough': 'first'  
}).reset_index()

# This line fixes your KeyError: 'tree_count'
tree_stats.columns = ['zip_code', 'tree_count', 'health_numeric', 'borough']

# Calculate % Good Health safely
health_counts = trees_alive.groupby(['postcode', 'health']).size().unstack(fill_value=0)
if 'Good' not in health_counts.columns: health_counts['Good'] = 0
health_counts['total'] = health_counts.sum(axis=1)
health_counts['percent_good'] = (health_counts['Good'] / health_counts['total']) * 100

# Merge the % Good metric back into tree_stats
tree_stats = tree_stats.merge(health_counts[['percent_good']], left_on='zip_code', right_index=True)

# --- 4. THE MASTER MERGE ---
final_df = pd.merge(tree_stats, income_df, on='zip_code').dropna(subset=['median_income', 'percent_good'])

print(f"Harmonization Complete: {len(final_df)} Zip Codes successfully linked.")
final_df.head()

In [ ]:
# Z-Score filter for outliers
from scipy import stats
import numpy as np

# Remove Zip Codes with very few trees (e.g., less than 50)
filtered_df = final_df[final_df['tree_count'] > 50].copy()

# Filter out extreme income outliers (more than 3 standard deviations away)
z_scores = np.abs(stats.zscore(filtered_df['median_income']))
clean_df = filtered_df[z_scores < 3]

print(f"Removed {len(final_df) - len(clean_df)} outlier Zip Codes.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create the scatter plot with a regression line
plt.figure(figsize=(12, 8))

# Use 'health_numeric' 
# Also using clean_df here makes the graph much more readable!
sns.regplot(data=clean_df, x='median_income', y='health_numeric', 
            scatter_kws={'alpha':0.5}, line_kws={'color':'red'})

plt.title('Relationship Between Neighborhood Wealth and Tree Health (NYC)', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('Average Tree Health Score (1=Poor, 3=Good)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Calculate the correlation matrix using the cleaned data
# We use 'health_numeric' because that is the name defined in our harmonization block
correlation = clean_df['median_income'].corr(clean_df['health_numeric'])

print(f"The Correlation between Income and Tree Health (Cleaned) is: {correlation:.2f}")

if correlation > 0.3:
    print("Result: There is a moderate to strong positive relationship!")
elif correlation > 0.1:
    print("Result: There is a weak positive relationship.")
elif correlation > -0.1:
    print("Result: There is virtually no linear relationship across the whole city.")
else:
    print("Result: There is a negative relationship.")

# Double check: Also look at the % Good Health correlation
corr_percent = clean_df['median_income'].corr(clean_df['percent_good'])
print(f"The Correlation for % Good Health is: {corr_percent:.2f}")

In [ ]:
# Scatter plot for Income vs. Total Tree Count
plt.figure(figsize=(12, 8))
sns.regplot(data=final_df, x='median_income', y='tree_count', 
            scatter_kws={'alpha':0.5}, line_kws={'color':'green'})

plt.title('Neighborhood Wealth vs. Number of Street Trees', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('Total Number of Trees', fontsize=12)

plt.show()

In [ ]:
# 1. Prepare health counts safely
# If 'status' is missing, it means we likely already filtered for 'Alive' trees earlier
if 'status' in trees_df.columns:
    health_counts_df = trees_df[trees_df['status'] == 'Alive'].copy()
else:
    health_counts_df = trees_df.copy()

# Ensure postcode is a string to match the census data zip_code
health_counts_df['postcode'] = health_counts_df['postcode'].astype(str)

# 2. Pivot the data to get Good/Fair/Poor counts per Zip
health_pivot = health_counts_df.groupby(['postcode', 'health']).size().unstack(fill_value=0)

# 3. Calculate the percentage
# We check if 'Good' exists in case a zip code has only 'Fair' or 'Poor' trees
if 'Good' in health_pivot.columns:
    health_pivot['total'] = health_pivot.sum(axis=1)
    health_pivot['percent_good'] = (health_pivot['Good'] / health_pivot['total']) * 100
else:
    health_pivot['percent_good'] = 0

# 4. MERGE: Add the new metric to final_df
# We drop the column first if it already exists to avoid '.x' or '.y' suffixes
if 'percent_good' in final_df.columns:
    final_df = final_df.drop(columns=['percent_good'])

final_df = final_df.merge(
    health_pivot[['percent_good']], 
    left_on='zip_code', 
    right_index=True, 
    how='left'
)

# 5. Plotting
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.regplot(data=final_df, x='median_income', y='percent_good', 
            scatter_kws={'alpha':0.4}, line_kws={'color':'green'})

plt.title('Neighborhood Wealth vs. Tree Health (% Good)', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('% of Trees in "Good" Health', fontsize=12)
plt.show()

In [ ]:
# We haven't found a correlations so far...
# Analyzing the whole city at once mixes concrete jungles (Manhattan) with garden boroughs (Queens). Lets try splitting the scatter plot by Borough.

# --- STEP 1: ADD BOROUGH INFO ---
# Create the lookup from the original trees_df 
zip_to_borough = trees_df.groupby('postcode')['borough'].first().reset_index()
zip_to_borough.columns = ['zip_code', 'borough']

# Merge it into final_df
if 'borough' not in final_df.columns:
    final_df = final_df.merge(zip_to_borough, on='zip_code', how='left')

# --- STEP 2: CREATE THE CLEAN VERSION ---
# We remove Zip Codes with < 20 trees and income outliers
from scipy import stats

# 1. Filter by tree count
clean_df = final_df[final_df['tree_count'] > 20].copy()

# 2. Filter by Income (removing top/bottom 1% or using Z-score)
z_scores = np.abs(stats.zscore(clean_df['median_income']))
clean_df = clean_df[z_scores < 3]

print(f"Cleaned data ready. Using {len(clean_df)} neighborhoods out of {len(final_df)}.")

# --- STEP 3: BOROUGH CORRELATION ---
print("\n--- Correlation by Borough (Wealth vs. % Good Health) ---")
for borough in clean_df['borough'].dropna().unique():
    subset = clean_df[clean_df['borough'] == borough]
    # We need at least a few data points to calculate correlation
    if len(subset) > 5:
        corr = subset['median_income'].corr(subset['percent_good'])
        print(f"{borough:15} | Correlation: {corr:+.2f} | Zip Codes: {len(subset)}")

# In the Bronx, we have a correlation of +0.46, which is statistically significant
# This suggests that in the Bronx, private investment or local stewardship could be related to socioeconomic status

In [ ]:
# This calculates the correlation for EACH borough separately
for borough in clean_df['borough'].unique():
    subset = clean_df[clean_df['borough'] == borough]
    corr = subset['median_income'].corr(subset['percent_good'])
    print(f"Correlation in {borough}: {corr:.2f}")

In [ ]:
# Create a side-by-side comparison of the Bronx and Manhattan
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Plot 1: The Bronx (The Strong Link)
bronx_data = clean_df[clean_df['borough'] == 'Bronx']
sns.regplot(data=bronx_data, x='median_income', y='percent_good', 
            ax=ax1, color='orange', line_kws={'color':'red'})
ax1.set_title(f'The Bronx: Strong Correlation (+0.46)\nWealth = Health', fontsize=14)

# Plot 2: Manhattan (The Concrete Jungle)
manhattan_data = clean_df[clean_df['borough'] == 'Manhattan']
sns.regplot(data=manhattan_data, x='median_income', y='percent_good', 
            ax=ax2, color='gray', line_kws={'color':'blue'})
ax2.set_title(f'Manhattan: No Correlation (-0.01)\nDensity Levels the Playing Field', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
species_diversity = (trees_df.groupby('postcode')['spc_common'].nunique().reset_index())
species_diversity.columns = ['zip_code', 'species_diversity']
# species_diversity.head()

In [ ]:
final_df = final_df.merge(species_diversity, on='zip_code', how='left')
print(final_df.nlargest(10, 'species_diversity')[['zip_code', 'borough', 'tree_count', 'species_diversity']])

In [ ]:
plt.figure(figsize=(12, 8))
sns.regplot(
    data=final_df,
    x='median_income',
    y='species_diversity',
    scatter_kws={'alpha': 0.5},
    line_kws={'color': 'red'}
)

plt.title('Median Income vs Tree Species Richness')
plt.xlabel('Median Household Income ($)')
plt.ylabel('Number of Unique Tree Species')
plt.show()

corr = final_df['median_income'].corr(final_df['species_diversity'])
print(f"Correlation between income and species richness: {corr:.2f}")

In [ ]:
zip_to_borough = trees_df.groupby('postcode')['borough'].first().reset_index()
zip_to_borough.columns = ['zip_code', 'borough']

if 'borough' not in final_df.columns:
    final_df = final_df.merge(zip_to_borough, on='zip_code', how='left')

# Calculate average and standard deviation of tree counts
mean_trees = final_df['tree_count'].mean()
std_trees = final_df['tree_count'].std()

# Keep zip codes with tree counts close to the average
close_avg_df = final_df[
    (final_df['tree_count'] >= mean_trees - std_trees) &
    (final_df['tree_count'] <= mean_trees + std_trees)
].copy()

print(f"Average tree count: {mean_trees:.2f}")
print(f"Standard deviation: {std_trees:.2f}")

print("\nCorrelation by Borough: Income vs Species Richness")
for borough in close_avg_df['borough'].dropna().unique():
    subset = close_avg_df[close_avg_df['borough'] == borough]
    
    if len(subset) > 5:
        corr = subset['median_income'].corr(subset['species_diversity'])
        print(f"{borough} Correlation: {corr:+.2f}")

In [ ]:
# Group by species and calculate stats
species_health_df = trees_df.groupby('spc_common').agg(
    tree_count=('tree_id', 'count'),
    avg_health_score=('health_numeric', 'mean')
).reset_index()

# Keep only species with a large enough sample so tiny groups don't distort things
species_health_df = species_health_df[species_health_df['tree_count'] >= 500]

# Sort by health score
species_health_df = species_health_df.sort_values(by='avg_health_score', ascending=False)

print(species_health_df.head(15))

In [ ]:
# Group by borough + species
species_borough_health = trees_df.groupby(['borough', 'spc_common']).agg(
    tree_count=('tree_id', 'count'),
    avg_health_score=('health_numeric', 'mean')
).reset_index()

# Keep only borough-species combos with enough trees
species_borough_health = species_borough_health[species_borough_health['tree_count'] >= 200]

print(species_borough_health.head())

In [ ]:
for borough in species_borough_health['borough'].unique():
    print(f"\n--- Top species in {borough} ---")
    top_species = species_borough_health[species_borough_health['borough'] == borough] \
        .sort_values(by='avg_health_score', ascending=False)
    
    print(top_species[['spc_common', 'tree_count', 'avg_health_score']].head(10))

In [ ]:
borough_name = 'Bronx'   # change this to Queens, Manhattan, Brooklyn, Staten Island

borough_data = species_borough_health[
    species_borough_health['borough'] == borough_name
].sort_values(by='avg_health_score', ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(data=borough_data, x='avg_health_score', y='spc_common')

plt.title(f'Top 15 Common Tree Species by Health Score in {borough_name}')
plt.xlabel('Average Health Score (1 = Poor, 3 = Good)')
plt.ylabel('Tree Species')
plt.show()

In [ ]:
# Average health of each species within each zip code
species_zip_health = trees_df.groupby(['postcode', 'spc_common']).agg(
    tree_count=('tree_id', 'count'),
    avg_health_score=('health_numeric', 'mean'),
    borough=('borough', 'first')
).reset_index()

species_zip_health = species_zip_health.rename(columns={'postcode': 'zip_code'})

# Merge with income
species_income_df = pd.merge(
    species_zip_health,
    income_df[['zip_code', 'median_income']],
    on='zip_code',
    how='inner'
)

print("\nCorrelation: Income vs Species Health Within Each Borough")

for borough in species_income_df['borough'].dropna().unique():
    print(f"\n=== {borough} ===")
    
    borough_subset = species_income_df[species_income_df['borough'] == borough]
    
    # Loop through species in this borough
    for species in borough_subset['spc_common'].unique():
        species_subset = borough_subset[borough_subset['spc_common'] == species]
        
        # Need enough zip codes for correlation to mean anything
        if len(species_subset) >= 10:
            corr = species_subset['median_income'].corr(species_subset['avg_health_score'])
            print(f"{species:25} | Correlation: {corr:+.2f} | Zip Codes: {len(species_subset)}")

In [ ]:
corr_data = []

for borough in species_income_df['borough'].dropna().unique():
    borough_subset = species_income_df[species_income_df['borough'] == borough]
    
    for species in borough_subset['spc_common'].unique():
        species_subset = borough_subset[borough_subset['spc_common'] == species]
        
        if len(species_subset) >= 10:
            corr = species_subset['median_income'].corr(
                species_subset['avg_health_score']
            )
            corr_data.append([borough, species, corr])

In [ ]:
corr_df = pd.DataFrame(corr_data, columns=['borough', 'species', 'correlation'])

heatmap_df = corr_df.pivot(index='species', columns='borough', values='correlation')

In [ ]:
plt.figure(figsize=(12, 20))

sns.heatmap(
    heatmap_df,
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)

plt.title('Correlation of Income vs Tree Health by Species and Borough')
plt.show()